In [1]:
import requests
import json
import pandas as pd
import re
import time
import sys
from bs4 import BeautifulSoup

# 1. 본문 텍스트 추출 함수
def get_blog_text(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/110.0.0.0"}
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, "html.parser")
        if soup.find("iframe", id="mainFrame"):
            real_url = "https://blog.naver.com" + soup.find("iframe", id="mainFrame")["src"]
            res_real = requests.get(real_url, headers=headers)
            soup_real = BeautifulSoup(res_real.text, "html.parser")
            content = soup_real.select_one(".se-main-container") or soup_real.select_one("#postViewArea")
            return content.get_text(" ", strip=True) if content else ""
    except: return ""
    return ""

# 2. 메인 수집 함수
def scrape_rextreme_all_pages():
    headers = {
        'X-Naver-Client-Id' : 'qlURw5tgE7CmccdW37fe', 
        'X-Naver-Client-Secret' : 'FOGoT6pEeT'
    }
    
    keywords = ["렉스트림", "rextreme"]
    total_items = []

    # [단계 1] 모든 페이지 목록 가져오기 (100건씩 끊어서 반복)
    print("🔎 검색 결과 목록을 전수 수집 중입니다...")
    for kw in keywords:
        # 네이버 API는 최대 1,000건까지 지원하므로 10번 반복 (100개씩)
        for i in range(10):
            start_pos = (i * 100) + 1
            url = f"https://openapi.naver.com/v1/search/blog.json?query={kw}&display=100&start={start_pos}&sort=date"
            res = requests.get(url, headers=headers)
            
            if res.status_code == 200:
                data = json.loads(res.text)
                items = data.get('items', [])
                if not items: break # 더 이상 가져올 데이터가 없으면 중단
                total_items.extend(items)
                
                # 전체 검색 결과 건수보다 많이 가져올 수 없으므로 체크
                if len(total_items) >= data.get('total', 0): break
            else:
                break
            time.sleep(0.1)

    # 중복 제거 (링크 기준)
    df = pd.DataFrame(total_items).drop_duplicates(subset=['link']).reset_index(drop=True)
    total_count = len(df)
    
    if total_count == 0:
        print("❌ 검색 결과가 없습니다.")
        return

    # [단계 2] 본문 수집 및 필터링
    print(f"🚀 총 {total_count}건의 블로그 분석 및 원문 저장을 시작합니다.")
    
    clean_results = []
    target_words = ['렉스트림', 'rextreme']
    exclude_words = ['주식회사', '인허가', '소재지', '면적', '취득일']

    for idx, row in df.iterrows():
        full_text = get_blog_text(row['link'])
        title = re.sub('<.*?>', '', row['title'])
        
        # 소문자로 변환하여 대소문자 상관없이 검사
        is_target = any(word in title.lower() or word in full_text.lower() for word in target_words)
        is_garbage = any(word in full_text for word in exclude_words)

        if is_target and not is_garbage:
            clean_results.append({
                'title': title,
                'postdate': row['postdate'],
                'fulltext': full_text,
                'link': row['link']
            })
        
        # 진행률 표시 (한 줄 업데이트)
        percent = ((idx + 1) / total_count) * 100
        sys.stdout.write(f"\r📊 진행도: {percent:6.2f}% | 처리중: {idx+1}/{total_count} | 저장 확정: {len(clean_results)}건")
        sys.stdout.flush()
        
        time.sleep(1.2)

    # [단계 3] 결과 저장
    final_df = pd.DataFrame(clean_results)
    final_df.to_csv('rextreme_naver_reviews.csv', index=False, encoding='utf-8-sig')
    print(f"\n\n✨ 완료! 총 {len(final_df)}건의 유효한 리뷰 데이터가 저장되었습니다.")

# 실행
scrape_rextreme_all_pages()

🔎 검색 결과 목록을 전수 수집 중입니다...
🚀 총 242건의 블로그 분석 및 원문 저장을 시작합니다.
📊 진행도: 100.00% | 처리중: 242/242 | 저장 확정: 136건

✨ 완료! 총 136건의 유효한 리뷰 데이터가 저장되었습니다.
